In [ ]:
"""
Author: Sophie A. Liu
Date: 06/12/2026 10:58am
Purpose: isolating local expression activity around each immunofluorescent labeled cell
"""

In [1]:
# importing necessary libraries
import pandas as pd
import numpy as np
from tqdm import tqdm 
import os

In [ ]:
#V = pd.read_csv("9NMF_isoC.csv")
V = pd.read_csv("9NMF_Apd1.csv")
#V = pd.read_csv("9NSF_isoC.csv")
#V = pd.read_csv("9NSF_Apd1.csv")

#S = pd.read_csv("iso_coordsNT.csv")      # spots from IF
S = pd.read_csv("apd1_coordsNT.csv")

In [ ]:
# setting parameters/ initializing things
v_coords = V[["x", "y"]].to_numpy()
s_coords = S[["x", "y"]].to_numpy()

gene_cols = V.columns[4:13]

radius = 40                                # balancing capturing enough cells but account for sparsity, cell ~ 8 microns
np.random.seed(42)                         # the answer to the ultimate question of life, the universe, and everything.

In [ ]:
print(gene_cols)

In [9]:
from scipy.spatial import cKDTree

In [10]:
def inputs(S, V, gene_cols, s_coords, v_coords):

    # KD-trees
    s_tree = cKDTree(s_coords)
    v_tree = cKDTree(v_coords)

    # encoding cell types as integers leads to faster processing
    type_map = {
        "tdtomato": 0,
        "gc3ai": 1,
        "cd8": 2,
        "lectin": 3
    }
    S_cells = np.array([type_map.get(x, -1) for x in S["cell_type"].values])

    # extracting gene matrix :)
    V_genes = V[gene_cols].to_numpy()

    return s_tree, v_tree, S_cells, V_genes

In [ ]:
# counts of each cell type in the neighborhood of a IF-labeled cell, as well as some derived metrics.
def counts_in_radius(center, s_tree, S_cells, radius):

    idx = s_tree.query_ball_point(center, r=radius)

    if len(idx) == 0:
        counts = np.zeros(4)   # for all four types, if nothing then set 0. Loops through all neighborhoods
    else:
        types = S_cells[idx]
        counts = np.bincount(types[types >= 0], minlength=4)   # our result: counts = [n_tdtomato, n_gc3ai, n_cd8, n_lectin]

    n_alive, n_dying, n_immune, n_endothelial = counts         # renaming the channels to what cell type they represent

    # calculating later metrics so I don't have to do it downstream
    n_tumor = n_alive + n_dying
    total = n_tumor + n_immune + n_endothelial
    exist_dying = 1 if n_alive > 0 else 0                      # binary for vis

    return counts, n_tumor, total, exist_dying

In [ ]:
# mean expression of each gene in the neighborhood of a IF-labeled cell, a better representation,
# spatial smoothing incorporated to account for sparsity effects
def get_gene_means(center, v_tree, V_genes, radius):
    idx = v_tree.query_ball_point(center, r=radius)

    if len(idx) == 0:
        return np.zeros(V_genes.shape[1])

    return V_genes[idx].mean(axis=0)

In [27]:
def append_row(center, s_tree, v_tree, S_types, V_genes, radius):

    counts, n_tumor, total, exist_dying = counts_in_radius(
        center, s_tree, S_types, radius
    )

    gene_means = get_gene_means(
        center, v_tree, V_genes, radius
    )

    row = np.concatenate([
        np.array([center[0], center[1]]),
        counts,
        np.array([n_tumor, total, exist_dying]),
        gene_means
    ])

    return row

In [50]:
# function for final assembly/joining of neighborhoodresults. didn't vectorize. 
def compute_neighborhoods(
    S, V, s_coords, v_coords, gene_cols, radius):

    s_tree, v_tree, S_types, V_genes = inputs(
        S, V, gene_cols, s_coords, v_coords
    )

    n_centers = len(s_coords)
    n_genes = V_genes.shape[1]

    results = np.zeros((n_centers, 9 + n_genes))

    for i, center in enumerate(tqdm(s_coords, desc="Processing")):
        results[i] = append_row(
            center, s_tree, v_tree, S_types, V_genes, radius
        )

    columns = (
        ["cx", "cy",
         "n_alive", "n_dying", "n_immune", "n_lectin",
         "n_tumor", "all", "exist_dying"]
        + list(gene_cols)
    )

    return pd.DataFrame(results, columns=columns)

In [59]:
# progress bar was for when I was doing individual genes and it took forever.
df = compute_neighborhoods(
    S=S,
    V=V,
    s_coords=s_coords,
    v_coords=v_coords,
    gene_cols=gene_cols,
    radius=radius
)

df = df.join(S[["cell_type", "sample"]])         # maintaining cell type and sample info for downstream

Processing: 100%|██████████| 70927/70927 [00:01<00:00, 37521.68it/s]


In [60]:
df_clean = df[df["n_tumor"] > 0]

In [53]:
# helps restore independence by sampling non-overlapping neighborhoods using a greedy algorithm.
# random hard-core thinning. Matern soft potentially better but this made the most sense to me
def non_overlapping(df, n, radius):
    coords = df[['cx', 'cy']].to_numpy()
    remaining_idx = np.arange(len(coords))

    selected_idx = []

    while len(selected_idx) < n and len(remaining_idx) > 0:
        # random seeded 42 point
        pick_i = rng.choice(remaining_idx)
        selected_idx.append(pick_i)

        tree = cKDTree(coords[remaining_idx])

        # getting rid of all other points in that radius
        neighbors = tree.query_ball_point(coords[pick_i], r=radius)
        to_remove = set(remaining_idx[neighbors])

        remaining_idx = np.array([i for i in remaining_idx if i not in to_remove])

    return df.iloc[selected_idx].copy()

In [61]:
df_sub = non_overlapping(df_clean, n = 1000,                   # 3397 tot, choose >383 so we good
                             radius=radius*2)    

In [62]:
# verify output
print(df_sub.iloc[300:305, ])
df_sub.shape

                cx           cy  n_alive  n_dying  n_immune  n_lectin  \
24155  2207.243822  7065.726431     16.0      0.0       0.0       0.0   
57330  8095.126643  9769.037471     13.0      0.0       0.0       0.0   
627    7407.899626  2710.878864     13.0      0.0       1.0       0.0   
62094  2178.334024  8296.924975     14.0      1.0       0.0       0.0   
36719  2751.080900  7909.239750      8.0      1.0       0.0       0.0   

       n_tumor   all  exist_dying             1             2             3  \
24155     16.0  16.0          1.0  1.055187e-05  5.263370e-06  1.111714e-05   
57330     13.0  13.0          1.0  6.778232e-07  7.977626e-08  1.078930e-07   
627       13.0  14.0          1.0  3.643438e-07  4.384586e-08  1.723786e-07   
62094     15.0  15.0          1.0  5.458123e-06  8.950799e-07  7.257788e-06   
36719      9.0   9.0          1.0  6.414876e-06  3.222835e-06  7.633950e-06   

                  4             5             6             7             8  \
24155  

(1000, 20)

In [63]:
df_sub.to_csv("0613_9NMF_pd140.csv", index=False)